In [ ]:
import numpy as np
import pandas as pd
import mlflow.xgboost
from scipy.optimize import minimize

# === Load the trained XGBoost model ===
logged_model_uri = 'runs:/model_run_0010/model'
model = mlflow.xgboost.load_model(logged_model_uri)

# === Define model input features ===
model_features = [
    'process_signal_0032',
    'process_signal_0018',
    'process_signal_0004',
    'process_signal_0069',
    'process_signal_0081',
    'process_signal_0094',
    'process_signal_0025',
    'process_signal_0015',
    'process_signal_0045',
    'process_signal_0016',
    'process_signal_0102',
    'process_signal_0067',
    'process_signal_0012',
    'process_signal_0023',
    'process_signal_0051'
]

# === Indexing for optimization variables ===
var_index = {
    'Tfg': 0, 'Tair': 1, 'Mdfg': 2, 'H2': 3,
    'AAS': 4, 'CO': 5, 'CO2': 6, 'ASH': 7,
    'ESP_ASH': 8, 'TM': 9, 'AF': 10, 'GCV': 11
}
additional_index = {
    'MILL_REJ': 12, 'BLOW_DOWN': 13, 'ESP_RAD': 14,
    'PA_FAN': 15, 'COAL_CONS': 16, 'NOX': 17,
    'ESP_TEMP': 18, 'ECO_TEMP': 19, 'SUPR_HTR': 20, 'WIND_BOX': 21,
    'REJECT_QUA': 22, 'REJECT_GCV': 23, 'MAKEUP_WATER': 24,
    'STEAM_GEN': 25, 'STEAM_ENTHALPY': 26, 'FEEDW_ENTHALPY': 27
}

# === Bounds for optimization ===
bounds = [
    (120, 150), (15, 60), (5, 9), (2, 4), (0, 8), (0, 10),
    (10, 16), (0, 30), (0, 100), (11, 15), (0.002, 0.3), (3500, 4000),
    (0.03, 0.06), (0.22, 1.17), (1.12, 1.75),
    (15.07, 31.26), (0.19, 0.37), (80.20, 296.96),
    (101.65, 185.22), (290.00, 349.98),
    (369.80, 436.67), (15.20, 49.99),
    (1000, 1300), (1300, 1500), (10, 100),
    (50, 200), (750, 1000), (100, 200)
]

# === Loss calculation function ===
def calculate_losses(x):
    Tfg, Tair, Mdfg, H2, AAS, CO, CO2, ASH, ESP_ASH, TM, AF, GCV = x[:12]
    RejectQuality = x[22]
    RejectGCV = x[23]
    MakeUpWater = x[24]
    SteamGen = x[25]
    SteamEnthalpy = x[26]
    FeedwEnthalpy = x[27]

    L1 = ((Mdfg * 0.24) * (Tfg - Tair) / GCV) * 100
    L2 = (TM * (584 + 0.45 * (Tfg - Tair))) / GCV
    L3 = 8.937 * (584 + 0.45 * (Tfg - Tair)) * H2 / GCV
    L4 = (AF * AAS * 0.45 * (Tfg - Tair) * 100) / GCV
    L5 = 0.02
    L6 = 1.13
    L7 = ((ASH * 15 / 100) * 10 * 8077) / (100 * GCV)
    L8 = ((ASH * 80 / 10000) * 0.23 * (Tfg - Tair) * 100) / GCV
    L9 = ((ASH * 20 / 10000) * 0.23 * (Tfg - Tair) * 100) / GCV
    L10 = 0.65
    L11 = ((0.8 * MakeUpWater * 300 + 0.2 * MakeUpWater * 150) * 100) / ((SteamGen * (SteamEnthalpy - FeedwEnthalpy)) / 0.85)
    L12 = (RejectQuality * RejectGCV * 100) / ((SteamGen * (SteamEnthalpy - FeedwEnthalpy)) * 1000 / 0.85)

    UL = L6 + L7
    SL = L8 + L9
    total = L1 + L2 + L3 + L4 + L5 + UL + SL + L10 + L11 + L12

    return total, {
        'Total Loss': total, 'L1': L1, 'L2': L2, 'L3': L3, 'L4': L4, 'L5': L5,
        'L6': L6, 'L7': L7, 'L8': L8, 'L9': L9, 'L10': L10, 'L11': L11, 'L12': L12
    }

# === Monte Carlo Loss ===
def calculate_loss_ranges(n_samples=5000):
    'Calculate min/max for each loss component through Monte Carlo sampling'
    loss_ranges = {f'L{i}': [np.inf, -np.inf] for i in range(1,13)}
    loss_ranges['Total Loss'] = [np.inf, -np.inf]

    for _ in range(n_samples):
        sample = [np.random.uniform(low, high) for (low, high) in bounds]
        try:
            total_loss, loss_dict = calculate_losses(sample)
            for name, value in loss_dict.items():
                loss_ranges[name][0] = min(loss_ranges[name][0], value)
                loss_ranges[name][1] = max(loss_ranges[name][1], value)
        except Exception:
            continue
    return loss_ranges

# === Print Loss Ranges as (min, max) ===
def print_loss_ranges(loss_ranges):
    print('\n🔥 Loss Value Ranges (Min, Max):')
    for loss in ['Total Loss'] + [f'L{i}' for i in range(1,13)]:
        min_val, max_val = loss_ranges.get(loss, [0,0])
        print(f"{loss}: ({min_val:.3f}, {max_val:.3f})")

# === Main Execution ===
if __name__ == '__main__':
    # 1. Run optimization (your original code)
    n_runs = 20
    optimized_solutions = []
    for i in range(n_runs):
        x0 = np.array([np.random.uniform(low, high) for (low, high) in bounds])
        res = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=[
            {'type': 'ineq', 'fun': lambda x: x[var_index['Tfg']] - x[var_index['Tair']] - 10},
            {'type': 'ineq', 'fun': lambda x: (x[var_index['AAS']] / x[var_index['Mdfg']]) - 1.5},
            {'type': 'ineq', 'fun': lambda x: x[var_index['TM']] - x[var_index['H2']]}
        ])
        if res.success:
            optimized_solutions.append(res.x)

    if not optimized_solutions:
        raise RuntimeError('All optimization attempts failed.')

    optimized_array = np.array(optimized_solutions)
    print('\n📈 Optimal Input Variable Ranges (Based on Multiple Runs):')
    for i, (low, high) in enumerate(bounds):
        opt_min = optimized_array[:, i].min()
        opt_max = optimized_array[:, i].max()
        print(f"Var {i} - Optimum Min: {opt_min:.3f}, Max: {opt_max:.3f}, Original Bounds: ({low}, {high})")

    best_idx = np.argmax([-objective(x) for x in optimized_solutions])
    x_opt = optimized_solutions[best_idx]
    efficiency = -objective(x_opt)
    _, loss_dict = calculate_losses(x_opt)

    print(f"\n✅ Best Boiler Efficiency Achieved: {efficiency:.2f}%")
    print('\n📊 Optimal Input Variables:')
    for name, idx in var_index.items():
        print(f"{name}: {x_opt[idx]:.3f}")
    for name, idx in additional_index.items():
        print(f"{name}: {x_opt[idx]:.3f}")

    print('\n🔥 Loss Breakdown (Optimized Point):')
    for name, value in loss_dict.items():
        print(f"{name}: {value:.3f}")

    # === Print Loss Ranges Across Envelope as (min, max) ===
    loss_ranges = calculate_loss_ranges(2000)
    print_loss_ranges(loss_ranges)


In [ ]:
import numpy as np
import pandas as pd
import mlflow.xgboost
from scipy.optimize import minimize

# === Load the trained XGBoost model ===
logged_model_uri = 'runs:/model_run_0010/model'
model = mlflow.xgboost.load_model(logged_model_uri)

# === Define model input features ===
model_features = [
    'process_signal_0032',
    'process_signal_0018',
    'process_signal_0004',
    'process_signal_0069',
    'process_signal_0081',
    'process_signal_0094',
    'process_signal_0025',
    'process_signal_0015',
    'process_signal_0045',
    'process_signal_0016',
    'process_signal_0102',
    'process_signal_0067',
    'process_signal_0012',
    'process_signal_0023',
    'process_signal_0051'
]

# === Indexing for optimization variables ===
var_index = {
    'Tfg': 0, 'Tair': 1, 'Mdfg': 2, 'H2': 3,
    'AAS': 4, 'CO': 5, 'CO2': 6, 'ASH': 7,
    'ESP_ASH': 8, 'TM': 9, 'AF': 10, 'GCV': 11
}
additional_index = {
    'MILL_REJ': 12, 'BLOW_DOWN': 13, 'ESP_RAD': 14,
    'PA_FAN': 15, 'COAL_CONS': 16, 'NOX': 17,
    'ESP_TEMP': 18, 'ECO_TEMP': 19, 'SUPR_HTR': 20, 'WIND_BOX': 21,
    'REJECT_QUA': 22, 'REJECT_GCV': 23, 'MAKEUP_WATER': 24,
    'STEAM_GEN': 25, 'STEAM_ENTHALPY': 26, 'FEEDW_ENTHALPY': 27
}

# === Bounds for optimization ===
bounds = [
    (120, 150), (15, 60), (5, 9), (2, 4), (0, 8), (0, 10),
    (10, 16), (0, 30), (0, 100), (11, 15), (0.002, 0.3), (3500, 4000),
    (0.03, 0.06), (0.22, 1.17), (1.12, 1.75),
    (15.07, 31.26), (0.19, 0.37), (80.20, 296.96),
    (101.65, 185.22), (290.00, 349.98),
    (369.80, 436.67), (15.20, 49.99),
    (1000, 1300), (1300, 1500), (10, 100),
    (50, 200), (750, 1000), (100, 200)
]

# === Loss calculation function ===
def calculate_losses(x):
    Tfg, Tair, Mdfg, H2, AAS, CO, CO2, ASH, ESP_ASH, TM, AF, GCV = x[:12]
    RejectQuality = x[22]
    RejectGCV = x[23]
    MakeUpWater = x[24]
    SteamGen = x[25]
    SteamEnthalpy = x[26]
    FeedwEnthalpy = x[27]

    L1 = ((Mdfg * 0.24) * (Tfg - Tair) / GCV) * 100
    L2 = (TM * (584 + 0.45 * (Tfg - Tair))) / GCV
    L3 = 8.937 * (584 + 0.45 * (Tfg - Tair)) * H2 / GCV
    L4 = (AF * AAS * 0.45 * (Tfg - Tair) * 100) / GCV
    L5 = 0.02  # Updated to match your constraint
    L6 = 1.13
    L7 = ((ASH * 15 / 100) * 10 * 8077) / (100 * GCV)
    L8 = ((ASH * 80 / 10000) * 0.23 * (Tfg - Tair) * 100) / GCV
    L9 = ((ASH * 20 / 10000) * 0.23 * (Tfg - Tair) * 100) / GCV
    L10 = 0.40  # Updated to match your constraint
    L11 = ((0.8 * MakeUpWater * 300 + 0.2 * MakeUpWater * 150) * 100) / ((SteamGen * (SteamEnthalpy - FeedwEnthalpy)) / 0.85)
    L12 = (RejectQuality * RejectGCV * 100) / ((SteamGen * (SteamEnthalphy - FeedwEnthalpy)) * 1000 / 0.85)

    UL = L6 + L7
    SL = L8 + L9
    total = L1 + L2 + L3 + L4 + L5 + UL + SL + L10 + L11 + L12

    return total, {
        'Total Loss': total, 'L1': L1, 'L2': L2, 'L3': L3, 'L4': L4, 'L5': L5,
        'L6': L6, 'L7': L7, 'L8': L8, 'L9': L9, 'L10': L10, 'L11': L11, 'L12': L12,
        'UL': UL, 'SL': SL
    }

def loss_constraints(x):
    total_loss, loss_dict = calculate_losses(x)
    constraints = []
    
    
    constraints.append(4.81 - loss_dict['L1'])    
    constraints.append(2.15 - loss_dict['L2'])    
    constraints.append(4.17 - loss_dict['L3'])    
    constraints.append(0.14 - loss_dict['L4'])    
    constraints.append(0.02 - loss_dict['L5'])    
    
    UL = loss_dict['L6'] + loss_dict['L7']
    constraints.append(1.47 - UL)                 
    
    SL = loss_dict['L8'] + loss_dict['L9']
    constraints.append(0.29 - SL)                 
    
    constraints.append(0.40 - loss_dict['L10'])   
    constraints.append(0.07 - loss_dict['L11'])
    constraints.append(0.13 - loss_dict['L12'])
    return constraints

# === Objective Function ===
def objective(x):
    'Minimize total loss (maximize efficiency)'
    total_loss, _ = calculate_losses(x)
    return total_loss

# === Monte Carlo Loss Ranges ===
def calculate_loss_ranges(n_samples=5000):
    'Calculate min/max for each loss component through Monte Carlo sampling'
    loss_ranges = {f'L{i}': [np.inf, -np.inf] for i in range(1,13)}
    loss_ranges.update({'Total Loss': [np.inf, -np.inf], 'UL': [np.inf, -np.inf], 'SL': [np.inf, -np.inf]})

    for _ in range(n_samples):
        sample = [np.random.uniform(low, high) for (low, high) in bounds]
        try:
            total_loss, loss_dict = calculate_losses(sample)
            for name, value in loss_dict.items():
                if name in loss_ranges:
                    loss_ranges[name][0] = min(loss_ranges[name][0], value)
                    loss_ranges[name][1] = max(loss_ranges[name][1], value)
        except Exception:
            continue
    return loss_ranges

# === Print Loss Ranges as (min, max) ===
def print_loss_ranges(loss_ranges):
    print('\n🔥 Loss Value Ranges (Min, Max):')
    for loss in ['Total Loss'] + [f'L{i}' for i in range(1,13)] + ['UL', 'SL']:
        if loss in loss_ranges:
            min_val, max_val = loss_ranges[loss]
            print(f"{loss}: ({min_val:.3f}, {max_val:.3f})")

# === Print Constraint Validation ===
def print_constraint_validation(loss_ranges):
    'Check which constraints are satisfied across the range'
    design_limits = {
        'L1': 4.81, 'L2': 2.15, 'L3': 4.17, 'L4': 0.14, 'L5': 0.02,
        'UL': 1.47, 'SL': 0.29, 'L10': 0.40
    }
    
    print('\n🎯 Design Constraint Validation:')
    for loss_name, limit in design_limits.items():
        if loss_name in loss_ranges:
            min_val, max_val = loss_ranges[loss_name]
            if max_val <= limit:
                status = '✅ ALWAYS SATISFIED'
            elif min_val > limit:
                status = '❌ ALWAYS VIOLATED'
            else:
                status = '⚠️ SOMETIMES VIOLATED'
            print(f"{loss_name}: Range({min_val:.3f}, {max_val:.3f}) vs Limit({limit}) - {status}")

# === Main Execution ===
if __name__ == '__main__':
    print('🚀 Starting Constrained Boiler Optimization with Design Loss Limits...')
    
    # 1. Run constrained optimization
    n_runs = 20
    optimized_solutions = []
    constraint_violations = []
    
    for i in range(n_runs):
        x0 = np.array([np.random.uniform(low, high) for (low, high) in bounds])
        
        # Original constraints + Design Loss Constraints
        all_constraints = [
            {'type': 'ineq', 'fun': lambda x: x[var_index['Tfg']] - x[var_index['Tair']] - 10},
            {'type': 'ineq', 'fun': lambda x: (x[var_index['AAS']] / x[var_index['Mdfg']]) - 1.5},
            {'type': 'ineq', 'fun': lambda x: x[var_index['TM']] - x[var_index['H2']]}
        ]
        
        # Add individual loss constraints
        for j in range(8):  # 8 loss constraints
            all_constraints.append({
                'type': 'ineq', 
                'fun': lambda x, idx=j: loss_constraints(x)[idx]
            })
        
        res = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=all_constraints)
        
        if res.success:
            optimized_solutions.append(res.x)
        else:
            # Check constraint violations for failed optimizations
            constraints_vals = loss_constraints(x0)
            constraint_violations.append(constraints_vals)

    print(f"\n📊 Optimization Results:")
    print(f"Successful optimizations: {len(optimized_solutions)}/{n_runs}")
    print(f"Failed optimizations: {len(constraint_violations)}")

    if optimized_solutions:
        optimized_array = np.array(optimized_solutions)
        print('\n📈 Optimal Input Variable Ranges (Constraint-Satisfying Solutions):')
        for i, (low, high) in enumerate(bounds):
            opt_min = optimized_array[:, i].min()
            opt_max = optimized_array[:, i].max()
            print(f"Var {i:2d} - Feasible Range: ({opt_min:7.3f}, {opt_max:7.3f}) | Original: ({low:7.2f}, {high:7.2f})")

        # Best solution
        best_idx = np.argmin([objective(x) for x in optimized_solutions])
        x_opt = optimized_solutions[best_idx]
        efficiency = 100 - objective(x_opt)
        _, loss_dict = calculate_losses(x_opt)

        print(f"\n✅ Best Constrained Boiler Efficiency: {efficiency:.2f}%")
        print(f"Total Loss: {objective(x_opt):.3f}%")

        print('\n🔥 Optimal Loss Breakdown:')
        design_limits = {'L1': 4.81, 'L2': 2.15, 'L3': 4.17, 'L4': 0.14, 'L5': 0.02, 'UL': 1.47, 'SL': 0.29, 'L10': 0.40}
        for name, value in loss_dict.items():
            if name in design_limits:
                limit = design_limits[name]
                status = '✅' if value <= limit else '❌'
                print(f"{status} {name}: {value:.3f}% (Limit: {limit}%)")
            elif name not in ['Total Loss']:
                print(f"ℹ️ {name}: {value:.3f}%")

    # 2. Calculate loss ranges across entire input space
    print('\n🔄 Calculating Loss Ranges Across Input Space...')
    loss_ranges = calculate_loss_ranges(10000)
    print_loss_ranges(loss_ranges)
        
    # 3. Validate constraints across ranges
    print_constraint_validation(loss_ranges)


In [ ]:
Loss Component Ranges:
Total Loss: (17.13, 18.93)%
L1 (Dry Flue Gas): (4.84, 5.35)% ❌ > 4.81% limit
L2 (H₂O in fuel): (1.78, 1.97)% ✅ < 2.15% limit  
L3 (H₂ in fuel): (2.89, 3.19)% ✅ < 4.17% limit
L4 (H₂O in Air): (0.49, 0.54)% ❌ > 0.14% limit
L5 (Partial Combustion): (0.019, 0.021)% ✅ < 0.02% limit
UL (Total Unburnt): (1.83, 2.02)% ❌ > 1.47% limit
SL (Total Sensible): (0.13, 0.15)% ✅ < 0.29% limit
L10 (Radiation): (0.62, 0.68)% ❌ > 0.40% limit

In [ ]:
import numpy as np
from scipy.optimize import minimize
import mlflow.xgboost

# Load the trained ML model
logged_model_uri = 'runs:/model_run_0014/model'
model = mlflow.xgboost.load_model(logged_model_uri)

# Feature names (in correct order)
model_features = [
    'process_signal_0012',              # x[0]
    'process_signal_0030',                  # x[1]
    'process_signal_0035',                 # x[2]
    'process_signal_0094',            # x[3]
    'process_signal_0067',         # x[4]
    'process_signal_0023',        # x[5]
    'process_signal_0096', # x[6]
    'process_signal_0102',      # x[7]
    'process_signal_0045',            # x[8]
    'process_signal_0040',                 # x[9]
    'process_signal_0051',   # x[10]
    'process_signal_0054',            # x[11]
    'process_signal_0060',              # x[12]
    'process_signal_0020'                          # x[13]      
]

# Bounds
bounds = [
    (2.1, 2.8),     # Fuel-air ratio
    (150, 300),     # Total air flow
    (30, 100),      # Wind box dp
    (10, 50),       # PA fan primary air flow
    (10, 50),       # Wind box air flow
    (0.5, 1.5),     # Secondary to primary ratio
    (120, 300),     # Steam flow
    (380, 420),     # Superheater temp
    (290, 350),     # ECO gas temp
    (15.45, 90),    # Coal flow
    (240, 300),     # ESP duct gas temp
    (150, 250),     # FD water flow
    (110, 170),     # FD water temp
    (2, 6),
]

max_scc = 0.202

# Initial guess (midpoints)
x0 = [np.mean(b) for b in bounds]

# Objective function (maximize efficiency)
def objective(x):
    input_df = pd.DataFrame([x], columns=model_features)
    pred = model.predict(input_df)[0]
    return -pred  # maximize efficiency

def air_to_fuel_ratio(x):
    total_air = x[1]
    coal_flow = x[9]
    return (total_air / coal_flow) - 2.11

def scc_constraint(x):
    coal_flow = x[9]
    steam_gen = x[6]
    scc = coal_flow / steam_gen
    return max_scc - scc


constraints = [
    {'type': 'ineq', 'fun': lambda x: x[1] - 150},      # Total Air Flow ≥ 150
    {'type': 'ineq', 'fun': air_to_fuel_ratio},         # Air-to-fuel ratio ≥ 2.11
    {'type': 'ineq', 'fun': lambda x: x[9] - 15.45},     # Coal Flow ≥ 15.45
    {'type': 'ineq', 'fun': lambda x: x[6] - 120},       # Steam ≥ 120
    {'type': 'ineq', 'fun': lambda x: x[10] - 190},      # APH outlet ≥ 190
    {'type': 'ineq', 'fun': lambda x: x[2] - 10},        # Wind box dp ≥ 10
    {'type': 'ineq', 'fun': scc_constraint},             # SCC ≤ max_scc
]

# --- Predict BEFORE optimization ---
initial_eff = model.predict(np.array(x0).reshape(1, -1))[0]

# --- Run Optimization ---
res = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=constraints)

if not res.success:
    raise RuntimeError(f"Optimization failed: {res.message}")

x_opt = res.x
eff_opt = -res.fun  # flip sign

# === Output Section ===
print('\n=== Boiler Efficiency Optimization ===')
print(f"Initial Predicted Efficiency : {initial_eff:.4f} %")
print(f"Maximized Predicted Efficiency: {eff_opt:.4f} %")

print('\n=== Feature Values Before vs After ===')
for name, before, after in zip(model_features, x0, x_opt):
    print(f"{name:55s} | Before: {before:7.3f} → After: {after:7.3f}")

In [ ]:
import numpy as np
import pandas as pd
import mlflow.xgboost
from scipy.optimize import minimize

# Load trained ML model
logged_model_uri = 'runs:/model_run_0014/model'
model = mlflow.xgboost.load_model(logged_model_uri)

# Features and bounds (simplified for example)
model_features = ['Fuel_Air_Ratio', 'Total_Air_Flow', 'Coal_Flow', 'Steam_Flow']
bounds = [
    (2.1, 2.8),    # Fuel-air ratio
    (150, 300),    # Total air flow (TPH)
    (40, 65),      # Coal flow (TPH)
    (200, 320)     # Steam flow (TPH)
]

# Enhanced objective with adaptive penalties
def objective(x):
    input_df = pd.DataFrame([x], columns=model_features)
    efficiency = model.predict(input_df)[0]
    
    # Calculate losses and SCC
    total_loss, loss_dict = calculate_losses(x)
    scc = x[2]/x[3]  # Coal flow/Steam flow
    
    # Penalty weights
    loss_penalty = 1000
    scc_penalty = 500
    
    # Constraint violations
    violations = [
        max(0, loss_dict['L1'] - 4.81),
        max(0, loss_dict['L2'] - 2.15),
        max(0, scc - 0.202)
    ]
    
    return -(efficiency) + sum(v * loss_penalty for v in violations) + (scc * scc_penalty)

# Enhanced constraints
def operational_constraints(x):
    return [
        {'type': 'ineq', 'fun': lambda x: x[1] - 150},      # Min air flow
        {'type': 'ineq', 'fun': lambda x: (x[1]/x[2]) - 2.1},  # Air/fuel ratio
        {'type': 'ineq', 'fun': lambda x: 0.202 - (x[2]/x[3])}  # SCC ≤ 0.202
    ]

# Multi-start optimization
def optimize_scc(n_runs=50):
    best = None
    for _ in range(n_runs):
        x0 = [np.random.uniform(low, high) for (low, high) in bounds]
        res = minimize(objective, x0, method='SLSQP', bounds=bounds, 
                      constraints=operational_constraints(x0))
        if res.success and (best is None or res.fun < best.fun):
            best = res
    return best

# Data flow example
if __name__ == '__main__':
    # Sample input
    initial_params = [2.45, 225, 52.5, 250]
    print('=== Initial State ===')
    print(f"Params: {dict(zip(model_features, initial_params))}")
    print(f"Predicted Efficiency: {model.predict(pd.DataFrame([initial_params], columns=model_features))[0]:.2f}%")
    print(f"SCC: {initial_params[2]/initial_params[3]:.4f}")
    
    # Optimization
    result = optimize_scc()
    
    if result.success:
        # Sample output
        optimized_params = result.x
        print('\n=== Optimized State ===')
        print(f"Params: {dict(zip(model_features, optimized_params))}")
        print(f"Predicted Efficiency: {-result.fun:.2f}%")
        print(f"SCC: {optimized_params[2]/optimized_params[3]:.4f}")
        
        # Constraint validation
        print('\n=== Constraint Check ===')
        print(f"Air flow: {optimized_params[1]} ≥ 150")
        print(f"Air/fuel ratio: {optimized_params[1]/optimized_params[2]:.2f} ≥ 2.1")
        print(f"SCC: {optimized_params[2]/optimized_params[3]:.4f} ≤ 0.202")
    else:
        print('Optimization failed to find valid solution')


In [ ]:
import numpy as np
import pandas as pd
import mlflow.xgboost
from scipy.optimize import minimize

# Load the trained ML model
logged_model_uri = 'runs:/model_run_0014/model'
model = mlflow.xgboost.load_model(logged_model_uri)

# Feature names (in correct order)
model_features = [
    'process_signal_0012',
    'process_signal_0030',    'process_signal_0035',
    'process_signal_0094',
    'process_signal_0067',
    'process_signal_0023',
    'process_signal_0096',
    'process_signal_0102',
    'process_signal_0045',
    'process_signal_0040',
    'process_signal_0051',
    'process_signal_0054',
    'process_signal_0060',
    'process_signal_0020'
]

# Bounds for optimization variables
bounds = [
    (2.1, 2.8), (150, 300), (30, 100), (10, 50), (10, 50), (0.5, 1.5),
    (120, 300), (380, 420), (290, 350), (15.45, 90), (240, 300),
    (150, 250), (110, 170), (2, 6)
]

max_scc = 0.202

def objective(x):
    input_df = pd.DataFrame([x], columns=model_features)
    pred = model.predict(input_df)[0]
    return -pred  # maximize efficiency

def air_to_fuel_ratio(x):
    total_air = x[1]
    coal_flow = x[9]
    return (total_air / coal_flow) - 2.11

def scc_constraint(x):
    coal_flow = x[9]
    steam_gen = x[6]
    return max_scc - (coal_flow / steam_gen)

constraints = [
    {'type': 'ineq', 'fun': lambda x: x[1] - 150},
    {'type': 'ineq', 'fun': air_to_fuel_ratio},
    {'type': 'ineq', 'fun': lambda x: x[9] - 15.45},
    {'type': 'ineq', 'fun': lambda x: x[6] - 120},
    {'type': 'ineq', 'fun': lambda x: x[10] - 190},
    {'type': 'ineq', 'fun': lambda x: x[2] - 10},
    {'type': 'ineq', 'fun': scc_constraint},
]

def multi_start_optimization(n_runs=30):
    best_solution = None
    best_efficiency = -np.inf
    
    for _ in range(n_runs):
        x0 = [np.random.uniform(low, high) for (low, high) in bounds]
        res = minimize(objective, x0, method='SLSQP', 
                      bounds=bounds, constraints=constraints)
        
        if res.success and (-res.fun > best_efficiency):
            best_solution = res.x
            best_efficiency = -res.fun
            
    return best_solution, best_efficiency

if __name__ == '__main__':
    # Initial state
    x_initial = [np.mean(b) for b in bounds]
    eff_initial = model.predict(pd.DataFrame([x_initial], columns=model_features))[0]
    scc_initial = x_initial[9] / x_initial[6]

    # Multi-start optimization
    x_opt, eff_opt = multi_start_optimization(n_runs=50)
    
    if x_opt is not None:
        scc_opt = x_opt[9] / x_opt[6]
        
        print('=== Optimization Results ===')
        print(f"{'Metric':<25} | {'Initial':<10} | {'Optimized':<10}")
        print('-'*50)
        print(f"{'Boiler Efficiency (%)':<25} | {eff_initial:10.4f} | {eff_opt:10.4f}")
        print(f"{'Specific Coal Consumption':<25} | {scc_initial:10.4f} | {scc_opt:10.4f}")
        
        print('\n=== Key Parameter Changes ===')
        key_params = [0, 1, 6, 9, 13]  # Fuel-air, Air flow, Steam, Coal, O2
        for idx in key_params:
            name = model_features[idx]
            before = x_initial[idx]
            after = x_opt[idx]
            print(f"{name:<50} {before:7.3f} → {after:7.3f}")
    else:
        print('Optimization failed to find valid solutions')
